[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/02_scene_and_radiomap.ipynb)

# 02 — Scene and Radio Map

**Purpose.** Build the Sionna-RT scene, generate one radio map at the deployed
tilt, and compute the five KPIs from it. PROJECT.md section 19 Steps 4 and 5.

This notebook produces the **baseline** — the reference every reported
improvement in notebooks 05a, 05b and 06 is measured against. It is also the
first place the coordinate convention, the grid geometry and the KPI code meet,
so it is where a mismatch between them is cheapest to find.

**Inputs.** `data/external/simulation_map/`, `data/raw/gcell_conf.csv`, and the
training split from notebook 01.

**Outputs.** The baseline KPI vector and the figures for section 27.3 of the
report.

**Requires** `uv sync --extra rt`.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna_rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Build the cell-band table

The atomic unit of the decision variable. Its row order is the canonical
ordering of every theta vector in the project — record it with any result you
save, or the numbers cannot be mapped back to cells later.

In [ ]:
from src.data.load import load_cell_config, load_processed
from src.radio import cell_band, radiomap, scene

cells = load_cell_config(cfg)
table = cell_band.build_table(cells, cfg)

theta_0 = cell_band.current_tilt(table)
lower, upper = cell_band.tilt_bounds(table)
print(f"{len(table)} cell-band pairs = {table.gcell_id.nunique()} cells x {table.band.nunique()} bands")
table.head(10)

## 3. Load the scene

Once. Loading 3,753 meshes and building the acceleration structure costs far more
than a single solve, and notebook 03 will reuse this same scene for hundreds of
configurations.

In [ ]:
sc = scene.load_scene(cfg)
xmin, ymin, xmax, ymax = scene.scene_bounds(cfg)
print(f"scene extent: x [{xmin}, {xmax}]  y [{ymin}, {ymax}]")

## 4. Coordinate and angle sanity check

**Do this before spending a solve.** PROJECT.md section 5 fixes
`yaw = deg2rad(90 - azimuth)` and `pitch = deg2rad(-tilt)`, so a positive
downtilt must produce a *negative* pitch.

A sign error here does not raise. It produces a complete, plausible radio map
with every beam pointing at the sky, and an optimizer that converges confidently
on the wrong answer. See docs/adr/0004.

In [ ]:
from src.radio import geometry

orient = geometry.orientations(table, theta_0)
check = pd.DataFrame(
    {
        "azimuth_deg": table["azimuth"],
        "tilt_deg": theta_0,
        "yaw_rad": orient[:, 0],
        "pitch_rad": orient[:, 1],
    }
)
assert (check.loc[check.tilt_deg > 0, "pitch_rad"] < 0).all(), "downtilt must give negative pitch"
check.head()

## 5. Attach transmitters and solve at the baseline tilt

One transmitter per cell-band, in table order, so the returned radio map can be
mapped back to the table without guessing.

In [ ]:
sc = scene.add_transmitters(sc, table, cfg)
rsrp = radiomap.evaluate(theta_0, sc, table, cfg)
print(f"RSRP array: {rsrp.shape}  (cell-bands x grid cells)")
print(f"finite fraction: {np.isfinite(rsrp).mean():.1%}")

## 6. UE density on the same grid

From the **training** split. Density built from all records would let the test
measurements shape the objective the optimizer maximises.

The alignment assertion is not ceremony: the two arrays broadcast happily at
mismatched offsets, and the resulting Band Priority Score is plausible and
wrong.

In [ ]:
from src.data.ue_density import assert_grid_aligned, ue_density

train_df = load_processed(cfg, "train")
rho = ue_density(train_df, cfg)
assert_grid_aligned(rho, rsrp)

print(f"N_UE = {rho.sum():,.0f} over {len(rho):,} grid cells")
print(f"grid cells with no observations: {(rho == 0).mean():.1%}")

## 7. The baseline KPI vector

The reference for the whole project. Every improvement claimed in notebooks 05a,
05b and 06 is stated relative to these five numbers.

In [ ]:
from src.kpi import vector

baseline_kpis = vector.kpi_vector(rsrp, rho, table, cfg)
pd.Series(baseline_kpis).to_frame("baseline").loc[list(cfg.kpi.order)]

## 8. Sanity-check the simulation against MDT

The one place simulated and measured RSRP can be compared. They will not match
exactly — the simulation is a ray-tracing approximation of a real network — but
they must be *close enough that the KPIs mean something*.

Compare at the MDT positions, per serving cell. A large systematic offset points
at a transmit power or antenna-gain error; a large spread points at the material
properties or the ray-tracing depth. Either way, the KPIs are computed from this
simulation, so an uncorrected bias here propagates into every result.

In [ ]:
# TODO(1): map each MDT record to its grid cell and its serving cell-band row
# TODO(2): compare measured rsrp against the simulated value at that location
# TODO(3): report bias and spread per cell, not pooled — a per-cell offset is a
#          configuration error, a global one is a calibration error
# TODO(4): decide and record the acceptance threshold before looking at the result

## 9. Spatial maps

The visual counterpart of section 7. A KPI table says hole rate is 8%; these say
whether the holes are at the edge of the area or in the middle of the densest
UE cluster, which is a very different outcome for the same number.

In [ ]:
from src.evaluation import analysis
from src.kpi import coverage, serving

per_cell = serving.cell_rsrp(rsrp, table, cfg)
n_ov = coverage.overlap_neighbors(per_cell, serving.serving_cell(per_cell), cfg)
b_star = serving.dominant_band(rsrp, table)

analysis.rsrp_map(rsrp, cfg, title="Baseline RSRP")
analysis.coverage_map(rsrp, cfg)
analysis.overlap_map(n_ov, cfg)
analysis.ue_density_map(rho, cfg)
analysis.dominant_band_map(b_star, table, cfg)

## 10. Handoff checklist

- [ ] The angle check in section 4 passed: positive downtilt gives negative pitch.
- [ ] Cells and MDT both sit inside the scene extent.
- [ ] `cfg.radio.grid.cell_size_m` is fixed and recorded — changing it later
      invalidates every surrogate sample generated against it.
- [ ] `cfg.radio.ray_tracing` is fixed and recorded, for the same reason.
- [ ] The baseline KPI vector is saved to `reports/results/`.
- [ ] The simulation-vs-MDT comparison in section 8 was run and its result
      accepted, or the discrepancy is recorded as a known limitation.